In [ ]:
# Once config is validated, you can proceed with the pipeline
print("Next steps:")
print("1. Update configs/ElevatedMazeFood.yml with anterior_bodyparts, posterior_bodyparts, fps")
print("2. Clean up broken KPMS project directory:")
print(f"   rm -rf {get_kpms_project_dir(project_path)}")
print("3. Run the prepare step again:")
print(f"   python scripts/run_kpms.py \\")
print(f"       --project-path {project_path} \\")
print(f"       --config configs/ElevatedMazeFood.yml \\")
print(f"       --steps prepare")

## Section 6: Process and Analyze Tracked Data

Extract relevant tracking data from h5 files and perform kinematic analysis using KPMS functions.

In [ ]:
# After updating ElevatedMazeFood.yml, validate it here
import yaml

config_path = _REPO_ROOT / "configs" / "ElevatedMazeFood.yml"
if config_path.exists():
    with open(config_path) as f:
        config = yaml.safe_load(f)
    
    print("Current ElevatedMazeFood.yml config:")
    print(f"  bodyparts: {config.get('bodyparts', 'NOT SET')}")
    print(f"  use_bodyparts: {config.get('use_bodyparts', 'NOT SET')}")
    print(f"  anterior_bodyparts: {config.get('anterior_bodyparts', 'NOT SET')}")
    print(f"  posterior_bodyparts: {config.get('posterior_bodyparts', 'NOT SET')}")
    print(f"  fps: {config.get('fps', 'NOT SET')}")
    print(f"  pose_estimation_format: {config.get('pose_estimation_format', 'NOT SET')}")
    print(f"  pose_file_extension: {config.get('pose_file_extension', 'NOT SET')}")
    
    # Validate
    if "anterior_bodyparts" not in config or "posterior_bodyparts" not in config:
        print("\n⚠️  anterior_bodyparts and/or posterior_bodyparts are MISSING!")
        print("These are required by keypoint-moseq.")
else:
    print(f"Config file not found at {config_path}")

## Section 5: Validate Configuration Parameters

Verify that all bodyparts specified in the configuration exist in your h5 files and that fps matches your video acquisition settings.

In [ ]:
# Once you have bodyparts from above, update your YAML config
# Example from your notebook:
# anterior_bodyparts = ["nose"]
# posterior_bodyparts = ["spine4"]
# use_bodyparts = ["spine4", "spine3", "spine2", "spine1", "head", "nose", "right ear", "left ear"]
# fps = 30

print("To fix your config:")
print("1. From the output above, identify the bodyparts in your h5 files")
print("2. Edit configs/ElevatedMazeFood.yml and add these fields:")
print("")
print("anterior_bodyparts:")
print("  - nose  # or appropriate anterior point")
print("")
print("posterior_bodyparts:")
print("  - spine4  # or appropriate posterior point")
print("")
print("fps: 30  # adjust to your video acquisition rate")
print("")
print("These are required by keypoint-moseq for skeletal model constraints.")

## Section 4: Update KPMS Configuration

Call `kpms.update_config()` with anterior_bodyparts, posterior_bodyparts, use_bodyparts, and fps parameters. Ensure your ElevatedMazeFood.yml is updated with these missing fields.

In [ ]:
# Try loading with keypoint-moseq to see what it detects
print("Attempting to load keypoints with keypoint-moseq...")

try:
    coordinates, confidences, bodyparts = kpms.load_keypoints(
        str(pose_data_dir),
        format="deeplabcut",
        extension="h5",
        recursive=True,
    )
    print(f"✓ Successfully loaded {len(coordinates)} recording(s)")
    print(f"Bodyparts detected: {bodyparts}")
    print(f"Coordinates shape per recording (e.g., first): {coordinates[0].shape if coordinates else 'N/A'}")
    print(f"Confidences shape per recording (e.g., first): {confidences[0].shape if confidences else 'N/A'}")
except Exception as e:
    print(f"✗ Error loading with KPMS: {e}")
    print("\nTrying manual h5 inspection instead...")

## Section 3: Configure KPMS for H5 Data

Set up the project directory and video directory paths. Identify available bodyparts from the h5 file metadata.

In [ ]:
# Set project paths
project_path = Path("/Users/atanugiri/Downloads/dlc-pose-estimation/ElevatedMazeFood-Atanu-2026-04-04")
pose_data_dir = get_pose_data_dir(project_path, use_filtered=False)

print(f"Project: {project_path}")
print(f"Pose data dir: {pose_data_dir}")
print()

# List h5 files
h5_files = sorted(pose_data_dir.glob("*.h5"))
print(f"Found {len(h5_files)} h5 file(s):")
for f in h5_files:
    print(f"  - {f.name}")

if not h5_files:
    print("No h5 files found!")
else:
    print(f"\nInspecting first file: {h5_files[0].name}")
    with h5py.File(h5_files[0], "r") as f:
        print(f"H5 structure:")
        def print_structure(name, obj):
            indent = "  " * name.count("/")
            if isinstance(obj, h5py.Dataset):
                print(f"{indent}{name}: shape={obj.shape}, dtype={obj.dtype}")
            elif isinstance(obj, h5py.Group):
                print(f"{indent}{name}/ (group)")
        f.visititems(print_structure)
        
        # Check for 'keypoints' dataset (common in DLC h5 exports)
        if "keypoints" in f:
            print(f"\n'keypoints' dataset: shape={f['keypoints'].shape}")
            print(f"Attributes: {list(f['keypoints'].attrs.keys())}")
        
        # Print all attributes
        print(f"\nFile-level attributes:")
        for key, val in f.attrs.items():
            print(f"  {key}: {val}")

## Section 2: Load and Inspect H5 Files

Load h5 files using h5py and inspect their structure, datasets, and attributes to understand the data format.

In [ ]:
import h5py
import keypoint_moseq as kpms
import numpy as np
from pathlib import Path
import sys

# Add repo root to path for kpms_utils
_REPO_ROOT = Path("/Users/atanugiri/Downloads/kpms_analysis")
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

from kpms_utils.path_utils import get_kpms_project_dir, get_pose_data_dir

print("Libraries imported successfully!")

## Section 1: Import Required Libraries

Import KPMS, h5py, and other necessary libraries for data analysis.

# Inspect DLC H5 Files and Configure KPMS

This notebook loads and inspects h5 files from your ElevatedMazeFood DLC project, extracts bodypart names, and ensures your KPMS configuration is correct.
